# Mechanics Tutorial 3: Coupling with Lumped Blood Circuits
![Pressure Volume Loop](3d0d-pv-loop.gif)

This tutorial shows how to couple 3d chamber models with 0d fluid models.

## Introduction

In this tutorial we will reproduce a simplified version of the model presented by [RegSalAfrFedDedQar:2022:cem](@citet).

> **Warning**
>
> The API for 3D-0D coupling is work in progress and is hence subject to potential breaking changes.

## Commented Program
We start by loading Thunderbolt and LinearSolve to use a custom direct solver of our choice.

In [1]:
using Thunderbolt, LinearSolve

Finally, we try to approach a valid initial state by solving a simpler model first.

In [2]:
using OrdinaryDiffEqTsit5, OrdinaryDiffEqOperatorSplitting

fluid_model_init = RSAFDQ2022LumpedCicuitModel()
u0 = zeros(Thunderbolt.num_states(fluid_model_init))
Thunderbolt.default_initial_state!(u0, fluid_model_init)
prob = ODEProblem((du, u, p, t) -> Thunderbolt.lumped_driver!(du, u, t, [], p), u0, (0.0, 100*fluid_model_init.THB), fluid_model_init)
sol = solve(prob, Tsit5())

retcode: Success
Interpolation: specialized 4th order "free" interpolation
t: 11931-element Vector{Float64}:
     0.0
     0.04838838098003728
     0.2792640643984848
     0.7267523496016612
     1.3438604381655068
     2.1442588892106365
     3.1555531172939957
     4.39824640576444
     5.90526888675417
     7.7164024194879755
     ⋮
 79949.53913317
 79956.29767348843
 79963.05655804648
 79969.81586426416
 79976.57547167127
 79983.33520206198
 79990.09491709921
 79996.85455341336
 80000.0
u: 11931-element Vector{Vector{Float64}}:
 [65.0, 120.0, 65.0, 145.0, 10.66, 4.0, 4.67, 3.2, 0.0, 0.0, 0.0, 0.0]
 [64.99775599207983, 120.00227770768569, 64.99411468494874, 145.00593718603966, 10.659998698778578, 3.9999999665140544, 4.669999657362968, 3.199999933568232, 0.0004815730430550299, 0.002134255662419863, 0.001058704949551402, 0.0013873724257603324]
 [64.98801864146336, 120.01307519006716, 64.96751009486455, 145.03415042166324, 10.65995732473332, 3.9999989362827226, 4.66998887341767, 3.1999

#plot(sol, idxs=[1,2,3,4], tspan=(99*fluid_model_init.THB, 100*fluid_model_init.THB))

In [3]:
# Precomputed initial guess
u₀fluid = sol.u[end]
@info "Total blood volume: $(sum(u₀fluid[1:4])) + $(fluid_model_init.Csysₐᵣ*u₀fluid[5]) + $(fluid_model_init.Csysᵥₑₙ*u₀fluid[6]) + $(fluid_model_init.Cpulₐᵣ*u₀fluid[7]) + $(fluid_model_init.Cpulᵥₑₙ*u₀fluid[8])"

[ Info: Total blood volume: 393.1882995055577 + 101.66387622657534 + 4788.817752623138 + 362.129558445206 + 379.8535603250976


We now generate the mechanical subproblem as in the first tutorial

In [4]:
scaling_factor = 3.7;

> **Warning**
>
> Tuning parameter until all bugs are fixed in this tutorial :)

In [5]:
mesh = generate_ideal_lv_mesh(16,4,19;
    inner_radius = scaling_factor*0.7,
    outer_radius = scaling_factor*1.0,
    longitudinal_upper = 0.4,
    apex_inner = scaling_factor* 1.3,
    apex_outer = scaling_factor*1.5,
    with_valvular_plane = true,
)

coordinate_system = compute_lv_coordinate_system(mesh; subdomains = ["myocardium"])
microstructure    = create_microstructure_model(
    coordinate_system,
    LagrangeCollection{1}()^3,
    ODB25LTMicrostructureParameters();
    subdomains = ["myocardium"],
);
passive_material_model = Guccione1991PassiveModel()
active_material_model  = Guccione1993ActiveModel()
function calcium_profile_function(x::LVCoordinate,t_global)
    linear_interpolation(t,y1,y2,t1,t2) = y1 + (t-t1) * (y2-y1)/(t2-t1)
    ca_peak(x)                          = 1.0
    t = t_global % 800.0
    if 0 ≤ t ≤ 120.0
        return linear_interpolation(t,        0.0, ca_peak(x),   0.0, 120.0)
    elseif t ≤ 272.0
        return linear_interpolation(t, ca_peak(x),        0.0, 120.0, 272.0)
    else
        return 0.0
    end
end
calcium_field = AnalyticalCoefficient(
    calcium_profile_function,
    coordinate_system,
)
sarcomere_model = CaDrivenInternalSarcomereModel(ConstantStretchModel(), calcium_field)
active_stress_model = ActiveStressModel(
    passive_material_model,
    active_material_model,
    sarcomere_model,
    microstructure,
)
weak_boundary_conditions = (RobinBC(1.0, "Epicardium"),NormalSpringBC(100.0, "Base"))
solid_model = QuasiStaticModel(:displacement, active_stress_model, weak_boundary_conditions);

The mesh was generated with a valvular plane, a meshed cap closing the basal orifice, so that the
chamber volume is the divergence-theorem integral over a *closed* surface and stays the cavity
volume however the base moves. The cap stands for the closed mitral valve together with its
annulus, and the chamber pressure of the coupler below acts on its ventricular face, so it has to
carry that pressure without bulging: it is a *stiff* isotropic closure, `α = 16000` giving a
Young's modulus of about 80 MPa, four orders of magnitude above the myocardial small-strain shear
modulus -- a fibrous rather than a muscular structure. At peak systolic pressure (≈ 14.6 kPa here)
that holds the cap's deflection relative to the annulus ring to ≈ 1.3 % of the cavity radius.

`mpU` is set equal to `α`, i.e. a Poisson ratio of ≈ 0.29: the cap is a closure and not tissue, so
there is no reason to make it nearly incompressible, and a compressible one spares the single
element the plate is thick from volumetric locking. There is no active stress and no
microstructure.

> **Warning**
>
> A cap at tissue stiffness is by far the most compliant part of the chamber. It then takes up
> the volume change that the wall should be doing, and the ventricle never builds systolic
> pressure at all -- the mitral valve of the 0D circuit stays open and the chamber just fills.

In [6]:
valvular_plane_model = QuasiStaticModel(
    :displacement,
    PK1Model(
        BioNeoHookean(; α = 16000.0, mpU = SimpleCompressionPenalty(16000.0)),
        NoMicrostructureModel(),
    ),
);

The solid model is now couple with the circuit model by adding a Lagrange multipliers constraining the 3D chamber volume to match the chamber volume in the 0D model.

In [7]:
fluid_model = RSAFDQ2022LumpedCicuitModel(; lv_pressure_given = false)
coupler = LumpedFluidSolidCoupler(
    [
        ChamberVolumeCoupling(
            "LVChamberSurface",
            :Vₗᵥ,
            :pₗᵥ,
            :pₗᵥ,
        )
    ],
    :displacement,
)

LumpedFluidSolidCoupler(ChamberVolumeCoupling[ChamberVolumeCoupling("LVChamberSurface", :Vₗᵥ, :pₗᵥ, :pₗᵥ)], :displacement)

The structural model is keyed by the subdomain it lives on, exactly as for an uncoupled mechanics
problem. Note that the subdomain map goes *inside* the coupled model: `RSAFDQ2022Split` annotates
the whole 3D-0D problem, so it is not itself something that lives on a subdomain. The cap is a
subdomain of the coupled problem like the wall is: the tying facets are the whole closed surface,
so the chamber pressure acts on the cap too and the coupler is added to both models.

In [8]:
coupled_model = RSAFDQ2022Model(
    Dict("myocardium" => solid_model, "valvular-plane" => valvular_plane_model),
    fluid_model,
    coupler,
);

> **Todo**
>
> Once we figure out a nicer way to do this we should add more detailed docs here.

Now we semidiscretize the model spatially as usual with finite elements and annotate the model with a stable split.
There is no Dirichlet condition: the epicardial Robin spring already pins every rigid body mode,
and the base is left to the pericardial spring above rather than clamped, because the cap is what
makes the chamber volume well defined while the base moves.

In [9]:
spatial_discretization_method = FiniteElementDiscretization(
    Dict(:displacement => LagrangeCollection{1}()^3),
)
splitform = semidiscretize(
    RSAFDQ2022Split(coupled_model),
    spatial_discretization_method,
    mesh,
)

dt₀ = 1.0
dtvis = 10.0
tspan = (0.0, 3*800.0)

(0.0, 2400.0)

This speeds up the CI #hide

In [10]:
tspan = (0.0, 10.0);    #hide

The remaining code is very similar to how we use SciML solvers. The 3D-0D operator assembles
into a block matrix, structural block and chamber-pressure block, so the inner solver is a
Schur complement solve around a factorization of the structural block.

In [11]:
chamber_solver = HomotopyPathSolver(
    NewtonRaphsonSolver(;
        max_iter=10,
        tol=1e-2,
        inner_solver=SchurComplementLinearSolver(LinearSolve.UMFPACKFactorization())
    )
)
blood_circuit_solver = Tsit5()
timestepper = LieTrotterGodunov((chamber_solver, blood_circuit_solver))

LTG (HomotopyPathSolver{Thunderbolt.GenericLocalNonlinearSolver}(NewtonRaphsonSolver{Float64}(0.01, 10, SchurComplementLinearSolver{LinearSolve.UMFPACKFactorization{@NamedTuple{}}}(LinearSolve.UMFPACKFactorization{@NamedTuple{}}(true, true, NamedTuple())), Thunderbolt.DefaultProgressMonitor(), true, nothing, false), Thunderbolt.GenericLocalNonlinearSolver(10, 0.0001)) -> OrdinaryDiffEqTsit5.Tsit5{typeof(OrdinaryDiffEqCore.trivial_limiter!), typeof(OrdinaryDiffEqCore.trivial_limiter!), FastBroadcast.Serial}(OrdinaryDiffEqCore.trivial_limiter!, OrdinaryDiffEqCore.trivial_limiter!, FastBroadcast.Serial()))

The coupled problem publishes every quantity it holds by name, so the pre-paced circuit state can be
transferred one state at a time instead of by slicing the solution vector.

In [12]:
u₀ = create_initial_condition(splitform)
for (sym, val) in zip(Thunderbolt.state_symbols(fluid_model), u₀fluid)
    setvariable!(u₀, splitform, sym, val)
end

> **Tip**
>
> `solution_variable_names(splitform)` lists what is available here: the displacement field, the
> chamber pressure `:pₗᵥ` introduced by the coupler, and the twelve circuit states.

In [13]:
problem = OperatorSplittingProblem(splitform, u₀, tspan)
integrator = init(problem, timestepper, dt=dt₀, verbose=true; dtmax=10.0);

# f2 = Figure()
# axs = [
#     Axis(f2[1, 1], title="LV"),
#     Axis(f2[1, 2], title="RV"),
#     Axis(f2[2, 1], title="LA"),
#     Axis(f2[2, 2], title="RA")
# ]

# vlv = Observable(Float64[])
# plv = Observable(Float64[])

# vrv = Observable(Float64[])
# prv = Observable(Float64[])

# vla = Observable(Float64[])
# pla = Observable(Float64[])

# vra = Observable(Float64[])
# pra = Observable(Float64[])

# lines!(axs[1], vlv, plv)
# lines!(axs[2], vrv, prv)
# lines!(axs[3], vla, pla)
# lines!(axs[4], vra, pra)
# for i in 1:4
#     xlims!(axs[1], 0.0, 180.0)
#     ylims!(axs[1], 0.0, 180.0)
# end
# display(f2)

> **Todo**
>
> recover online visualization of the pressure volume loop

Now we can finally solve the coupled problem in time.
The displacement field and the chamber quantities are all reached by name, so the loop never touches
the internals of the split. The descriptor is looked up once, outside the loop.

In [14]:
io = ParaViewWriter("CM03_3d0d-coupling");
displacement = solution_variable(splitform, :displacement)
for (u, t) in TimeChoiceIterator(integrator, tspan[1]:dtvis:tspan[2])
    store_timestep!(io, t, mesh) do file
        Thunderbolt.store_timestep_field!(file, t, u, displacement)
    end

    # The chamber pressure is a genuine unknown of the coupled problem, and the chamber volume is a
    # state of the 0D circuit, so both can simply be read out by name.
    # Note that `:Vₗᵥ` is the *0D* volume; the tying constrains it to match the 3D chamber volume at
    # convergence, but the two differ along the homotopy path.
    @info "$t: pₗᵥ = $(getvariable(u, splitform, :pₗᵥ)), Vₗᵥ = $(getvariable(u, splitform, :Vₗᵥ))"
    # if t > 0.0
    #     append!(vlv.val, getvariable(u, splitform, :Vₗᵥ))
    #     append!(plv.val, getvariable(u, splitform, :pₗᵥ))
    #     notify(vlv)
    #     notify(plv)
    # end
    # TODO plot other chambers
end

[ Info: 0.0: pₗᵥ = 0.0, Vₗᵥ = 117.55337177271954
[ Info: 10.0: pₗᵥ = 2.142376718166945, Vₗᵥ = 120.6075680395755


> **Tip**
>
> If you want to see more details of the solution process launch Julia with Thunderbolt as debug module:
> ```
> JULIA_DEBUG=Thunderbolt julia --project --threads=auto my_simulation_runner.jl
> ```

---

*This notebook was generated using [Literate.jl](https://github.com/fredrikekre/Literate.jl).*